# Seaquest DQN — Member 2 Mid-Range Experiments
**CnnPolicy | ALE/Seaquest-v5 | 10 Experiments**

All models, plots, and CSVs are saved to your Google Drive automatically.

In [ ]:
# ── Cell 1: Mount Google Drive ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# All results will be saved here
DRIVE_DIR = '/content/drive/MyDrive/seaquest_kerie'

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Saving to: {DRIVE_DIR}')

In [ ]:
# ── Cell 2: Install dependencies ────────────────────────────────
!pip install -q stable-baselines3 gymnasium[atari] ale-py opencv-python tensorboard
!pip install -q autorom[accept-rom-license]
print('All dependencies installed.')

In [ ]:
# ── Cell 3: Imports ─────────────────────────────────────────────
import csv
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import ale_py
import torch
from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnNoModelImprovement

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── Cell 4: Config ───────────────────────────────────────────────
EXPERIMENTS = {
    1: dict(name='exp1_baseline_mid',
            learning_rate=0.0001, gamma=0.94, batch_size=64,
            exploration_initial_eps=1.0, exploration_final_eps=0.05,
            exploration_fraction=0.10,
            notes='Baseline mid — good balance'),
    2: dict(name='exp2_lr_bump',
            learning_rate=0.0002, gamma=0.94, batch_size=64,
            exploration_initial_eps=1.0, exploration_final_eps=0.05,
            exploration_fraction=0.10,
            notes='Slight lr bump — faster policy updates'),
    3: dict(name='exp3_higher_gamma',
            learning_rate=0.0001, gamma=0.95, batch_size=64,
            exploration_initial_eps=1.0, exploration_final_eps=0.05,
            exploration_fraction=0.10,
            notes='Higher gamma — values future rewards more'),
    4: dict(name='exp4_lower_eps_start',
            learning_rate=0.0003, gamma=0.94, batch_size=64,
            exploration_initial_eps=0.9, exploration_final_eps=0.05,
            exploration_fraction=0.10,
            notes='Lower eps_start — less initial randomness'),
    5: dict(name='exp5_lower_eps_end',
            learning_rate=0.0001, gamma=0.96, batch_size=64,
            exploration_initial_eps=1.0, exploration_final_eps=0.02,
            exploration_fraction=0.10,
            notes='Lower eps_end — more greedy late stage'),
    6: dict(name='exp6_faster_decay',
            learning_rate=0.0002, gamma=0.95, batch_size=64,
            exploration_initial_eps=1.0, exploration_final_eps=0.05,
            exploration_fraction=0.05,
            notes='Faster decay — earlier shift to exploit'),
    7: dict(name='exp7_slower_decay',
            learning_rate=0.0001, gamma=0.94, batch_size=64,
            exploration_initial_eps=1.0, exploration_final_eps=0.05,
            exploration_fraction=0.25,
            notes='Slower decay — prolonged exploration'),
    8: dict(name='exp8_mixed_mid',
            learning_rate=0.0004, gamma=0.95, batch_size=64,
            exploration_initial_eps=0.9, exploration_final_eps=0.02,
            exploration_fraction=0.10,
            notes='Mixed mid — combined moderate changes'),
    9: dict(name='exp9_moderate_decay_high_gamma',
            learning_rate=0.0003, gamma=0.96, batch_size=64,
            exploration_initial_eps=1.0, exploration_final_eps=0.05,
            exploration_fraction=0.15,
            notes='Moderate decay with higher gamma'),
    10: dict(name='exp10_best_of_mid',
             learning_rate=0.0005, gamma=0.95, batch_size=64,
             exploration_initial_eps=0.9, exploration_final_eps=0.02,
             exploration_fraction=0.20,
             notes='Best-of-mid — final mid-range candidate'),
}

ENV_ID          = 'ALE/Seaquest-v5'
TOTAL_TIMESTEPS = 1_000_000
BUFFER_SIZE     = 100_000
LEARNING_STARTS = 10_000
TARGET_UPDATE   = 1_000
N_STACK         = 4

print('Config loaded. Experiments:', list(EXPERIMENTS.keys()))

In [ ]:
# ── Cell 5: Helper functions ─────────────────────────────────────
def make_env(render_mode=None):
    def _init():
        env = gym.make(ENV_ID, render_mode=render_mode)
        env = AtariWrapper(env)
        return env
    vec_env = DummyVecEnv([_init])
    vec_env = VecFrameStack(vec_env, n_stack=N_STACK)
    return vec_env


def run_experiment(exp_id, total_timesteps=TOTAL_TIMESTEPS):
    cfg = EXPERIMENTS[exp_id]
    print(f"\n{'='*60}")
    print(f"  Experiment {exp_id:02d}: {cfg['name']}")
    print(f"  Notes: {cfg['notes']}")
    print(f"  lr={cfg['learning_rate']}  gamma={cfg['gamma']}  "
          f"batch={cfg['batch_size']}  eps_start={cfg['exploration_initial_eps']}  "
          f"eps_end={cfg['exploration_final_eps']}  eps_frac={cfg['exploration_fraction']}")
    print(f"{'='*60}")

    run_dir  = os.path.join(DRIVE_DIR, cfg['name'])
    log_dir  = os.path.join(run_dir, 'tensorboard')
    eval_dir = os.path.join(run_dir, 'eval')

    # Skip if already completed
    done_path = os.path.join(run_dir, 'dqn_model.zip')
    if os.path.isfile(done_path):
        print(f'  SKIPPED — already completed ({done_path})')
        return done_path

    os.makedirs(run_dir,  exist_ok=True)
    os.makedirs(log_dir,  exist_ok=True)
    os.makedirs(eval_dir, exist_ok=True)

    train_env = make_env()
    eval_env  = make_env()

    model = DQN(
        policy                  = 'CnnPolicy',
        env                     = train_env,
        learning_rate           = cfg['learning_rate'],
        gamma                   = cfg['gamma'],
        batch_size              = cfg['batch_size'],
        exploration_initial_eps = cfg['exploration_initial_eps'],
        exploration_final_eps   = cfg['exploration_final_eps'],
        exploration_fraction    = cfg['exploration_fraction'],
        buffer_size             = BUFFER_SIZE,
        learning_starts         = LEARNING_STARTS,
        target_update_interval  = TARGET_UPDATE,
        tensorboard_log         = log_dir,
        verbose                 = 1,
        optimize_memory_usage   = False,
    )

    early_stop = StopTrainingOnNoModelImprovement(
        max_no_improvement_evals=3, verbose=1
    )
    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path = eval_dir,
        log_path             = eval_dir,
        eval_freq            = 25_000,
        n_eval_episodes      = 5,
        deterministic        = True,
        render               = False,
        callback_after_eval  = early_stop,
    )

    model.learn(
        total_timesteps     = total_timesteps,
        callback            = eval_callback,
        tb_log_name         = cfg['name'],
        reset_num_timesteps = True,
    )

    # Save final model to Drive
    model_path = os.path.join(run_dir, 'dqn_model')
    model.save(model_path)
    print(f'  Model saved -> {model_path}.zip')

    train_env.close()
    eval_env.close()

    # Save reward plot and CSV to Drive
    eval_npz = os.path.join(eval_dir, 'evaluations.npz')
    if os.path.isfile(eval_npz):
        data         = np.load(eval_npz)
        timesteps    = data['timesteps']
        mean_rewards = data['results'].mean(axis=1)
        std_rewards  = data['results'].std(axis=1)

        plt.figure(figsize=(10, 5))
        plt.plot(timesteps, mean_rewards, label='Mean Reward')
        plt.fill_between(timesteps,
                         mean_rewards - std_rewards,
                         mean_rewards + std_rewards,
                         alpha=0.3, label='Std Dev')
        plt.xlabel('Timesteps')
        plt.ylabel('Eval Reward')
        plt.title(f"Exp {exp_id:02d}: {cfg['name']}\n"
                  f"lr={cfg['learning_rate']} gamma={cfg['gamma']} "
                  f"batch={cfg['batch_size']} eps_frac={cfg['exploration_fraction']}")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plot_path = os.path.join(run_dir, 'reward_plot.png')
        plt.savefig(plot_path)
        plt.show()
        plt.close()
        print(f'  Plot saved  -> {plot_path}')

        csv_path = os.path.join(run_dir, 'results.csv')
        with open(csv_path, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['timestep', 'mean_reward', 'std_reward'])
            for t, m, s in zip(timesteps, mean_rewards, std_rewards):
                writer.writerow([int(t), f'{m:.2f}', f'{s:.2f}'])
        print(f'  CSV saved   -> {csv_path}')

    return model_path + '.zip'


print('Functions ready.')

In [ ]:
# ── Cell 6: Run all 10 experiments ──────────────────────────────
# Each experiment saves model + plot + CSV to Google Drive immediately after finishing.
# If Colab disconnects and you rerun, completed experiments are skipped automatically.

results = {}
for exp_id in EXPERIMENTS:
    path = run_experiment(exp_id, total_timesteps=TOTAL_TIMESTEPS)
    results[exp_id] = path

print('\n' + '='*60)
print('All experiments complete. Saved to Google Drive:')
for exp_id, path in results.items():
    print(f'  Exp {exp_id:02d}: {path}')
print('='*60)

In [ ]:
# ── Cell 7: Summary comparison plot ─────────────────────────────
# Plots all 10 experiments on one chart for easy comparison

plt.figure(figsize=(14, 7))

for exp_id, cfg in EXPERIMENTS.items():
    eval_npz = os.path.join(DRIVE_DIR, cfg['name'], 'eval', 'evaluations.npz')
    if os.path.isfile(eval_npz):
        data         = np.load(eval_npz)
        timesteps    = data['timesteps']
        mean_rewards = data['results'].mean(axis=1)
        plt.plot(timesteps, mean_rewards, label=f"Exp {exp_id}: lr={cfg['learning_rate']} γ={cfg['gamma']}")

plt.xlabel('Timesteps')
plt.ylabel('Mean Eval Reward')
plt.title('All 10 Experiments — Mid-Range Comparison')
plt.legend(fontsize=8)
plt.grid(True)
plt.tight_layout()

summary_path = os.path.join(DRIVE_DIR, 'all_experiments_summary.png')
plt.savefig(summary_path)
plt.show()
print(f'Summary plot saved -> {summary_path}')